# V56 - Compose the screening winners

Pre-registered plan: replicate the word-only win under a fresh seed, then gate two blends (plus assessment, plus assessment and content) against it. Also screened: direct module-mention features, counting how often each module name or alias appears in a user chat. An adopted composition becomes the base for the full build. No adoption without a CI lower bound above the floor and a clean worst fold.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics.pairwise import cosine_similarity

SEED = 606
N_SPLITS = 5
GATE_FLOOR = 0.002
rng = np.random.RandomState(SEED)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v56")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

## Views including module mentions

Mention features count direct references to each module (name words and syllabus keywords) in the user chat. They test whether explicit intent beats latent topics.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
catalog = pd.read_csv(DATA / "modules_catalog.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
A = np.array([[num(blobs[u].get(q)) for q in questions] for u in list(train["user_id"]) + test_ids])
A = np.where(np.isnan(A), np.nanmean(A, axis=0), A)
A = (A - A.mean(0)) / (A.std(0) + 1e-9)
A_tr, A_te = A[:n_train], A[n_train:]

by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
all_users = list(train["user_id"]) + test_ids
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
low = [d.lower() for d in docs]

STOP = set("dengan yang dan di ke dari untuk pada adalah ini itu sebagai dalam juga akan ada oleh karena sangat".split())
mentions = np.zeros((len(all_users), len(MODS)))
for j, m in enumerate(MODS):
    row = catalog.set_index("module_id").loc[m] if m in set(catalog["module_id"]) else None
    keys = set(re.findall(r"[a-z]+", str(row["module_name"]).lower())) - STOP if row is not None else set()
    keys |= set(re.findall(r"[a-z]+", str(row["description_and_syllabus"]).lower())) - STOP if row is not None else set()
    keys = {k for k in keys if len(k) > 3}
    for i, d in enumerate(low):
        mentions[i, j] = sum(d.count(k) for k in keys)
print("mention hit rate:", round(float((mentions.sum(1) > 0).mean()), 3))
M_tr, M_te = mentions[:n_train], mentions[n_train:]
M_tr = (M_tr - M_tr.mean(0)) / (M_tr.std(0) + 1e-9)
M_te = (M_te - M_tr.mean(0)) / (M_tr.std(0) + 1e-9)

T1 = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs)
W_tr, W_te = T1[:n_train].toarray(), T1[n_train:].toarray()
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1).fit(docs + catalog["description_and_syllabus"].fillna("").tolist())
syll = [catalog.set_index("module_id").loc[m, "description_and_syllabus"] if m in set(catalog["module_id"]) else "" for m in MODS]
CM = cosine_similarity(vec.transform(docs[:n_train]), vec.transform(syll))
CM_te = cosine_similarity(vec.transform(docs[n_train:]), vec.transform(syll))

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

## Gated composition

Baseline is word-only chat. Each composition must clear the floor on the CI lower bound with no negative fold, or the baseline stands.

In [ ]:
def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def ridge_oof(Xtr, Xte=None, alpha=3.0):
    oof = np.zeros_like(Y)
    pred = np.zeros((len(Xte), len(MODS))) if Xte is not None else None
    fids = np.zeros(len(Xtr), int)
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fi, (tr, va) in enumerate(skf.split(Xtr, strata, fold_groups)):
        for j in range(len(MODS)):
            m = Ridge(alpha=alpha, random_state=SEED)
            m.fit(Xtr[tr], Y[tr, j])
            oof[va, j] = m.predict(Xtr[va])
            if pred is not None:
                pred[:, j] += m.predict(Xte) / N_SPLITS
        fids[va] = fi + 1
    return oof, pred, fids

oof_w, pred_w, fid = ridge_oof(W_tr, W_te)
base = ndcg5(Y, oof_w)
print(f"baseline word-only OOF: {base.mean():.5f}")
cands = {}
oof_a, pred_a, _ = ridge_oof(A_tr, A_te)
oof_m, pred_m, _ = ridge_oof(np.nan_to_num(M_tr), np.nan_to_num(M_te))
cands["+ assessment"] = (np.hstack([W_tr, A_tr]), np.hstack([W_te, A_te]))
cands["+ mentions"] = (np.hstack([W_tr, np.nan_to_num(M_tr)]), np.hstack([W_te, np.nan_to_num(M_te)]))
cands["+ assessment + mentions"] = (np.hstack([W_tr, A_tr, np.nan_to_num(M_tr)]), np.hstack([W_te, A_te, np.nan_to_num(M_te)]))
cands["+ assessment + content"] = (np.hstack([W_tr, A_tr, CM]), np.hstack([W_te, A_te, CM_te]))
print(f"mentions-only OOF: {ndcg5(Y, oof_m).mean():.5f} | assessment-only OOF: {ndcg5(Y, oof_a).mean():.5f}")

adopted = []
for name, (Xtr, Xte) in cands.items():
    oo, _, _ = ridge_oof(Xtr, Xte)
    d = ndcg5(Y, oo) - base
    lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
    wf = min(ndcg5(Y[fid == k], oo[fid == k]).mean() - base[fid == k].mean() for k in range(1, N_SPLITS + 1))
    ok = (lo > GATE_FLOOR) and (wf >= 0)
    print(f"{name}: delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f} {'ADOPT' if ok else 'reject'}")
    if ok:
        adopted.append((name, d.mean()))
print("adopted:", adopted if adopted else "none; baseline stands")